In [ ]:
# Step - 1: Import all required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Step - 2: Load the engineered IoT weather dataset
df = pd.read_csv('../data/engineered_iot_weather_dataset.csv')
df['Machine failure'] = df['Machine failure'].astype(int)

print("Shape:", df.shape)
print("Columns:", list(df.columns))
df.head()

In [ ]:
# Step 3: Understand dataset structure
# 9996 rows, 40 columns — IoT rolling features + weather rolling features
# Target: Machine failure (binary: 0 = normal, 1 = failure)

print("=" * 50)
print("DATASET OVERVIEW")
print("=" * 50)
print(f"Total rows       : {df.shape[0]}")
print(f"Total columns    : {df.shape[1]}")
print(f"Failures (1)     : {df['Machine failure'].sum()}")
print(f"Normal (0)       : {(df['Machine failure']==0).sum()}")
print(f"Failure rate     : {df['Machine failure'].mean()*100:.2f}%")
print(f"Null values      : {df.isnull().sum().sum()}")
print("=" * 50)

In [ ]:
# Step 4: Analyze individual failure types
# TWF=Tool Wear, HDF=Heat Dissipation, PWF=Power,
# OSF=Overstrain, RNF=Random

failure_flags = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']

print("=" * 50)
print("FAILURE TYPE BREAKDOWN")
print("=" * 50)
for flag in failure_flags:
    count = df[flag].sum()
    pct = count / len(df) * 100
    print(f"{flag}: {count} cases ({pct:.2f}%)")
print("=" * 50)
print(f"Total failures   : {df['Machine failure'].sum()}")

In [ ]:
# Step 5: Select IoT rolling mean columns and weather rolling mean columns
# We use _roll_mean columns for correlation — they represent
# the smoothed trend, less noisy than raw std/var

iot_mean_cols = [
    'Air temperature [K]_roll_mean',
    'Process temperature [K]_roll_mean',
    'Rotational speed [rpm]_roll_mean',
    'Torque [Nm]_roll_mean',
    'Tool wear [min]_roll_mean'
]

weather_mean_cols = [
    'avg_temp_c_roll_mean',
    'min_temp_c_roll_mean',
    'max_temp_c_roll_mean',
    'precipitation_mm_roll_mean',
    'avg_sea_level_pres_hpa_roll_mean'
]

# Combine for correlation analysis
corr_cols = iot_mean_cols + weather_mean_cols + ['Machine failure']
print("Columns selected for correlation:", len(corr_cols))

In [ ]:
# Step 6: Generate correlation heatmap
# This shows relationships between IoT sensors, weather features,
# and machine failure — key insight for Track 2 EDA

plt.figure(figsize=(14, 10))
corr_matrix = df[corr_cols].corr()

sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    square=True,
    linewidths=0.5
)
plt.title('Correlation Matrix: IoT Sensors vs Weather Features',
          fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../plots/correlation_heatmap_iot_weather.png', dpi=150)
plt.show()

print("Saved to plots/correlation_heatmap_iot_weather.png")

In [ ]:
# Step 7: Find which features correlate most with Machine failure
# This tells us what the model will find most useful

failure_corr = corr_matrix['Machine failure'].drop('Machine failure')
failure_corr_sorted = failure_corr.abs().sort_values(ascending=False)

print("=" * 50)
print("TOP FEATURE CORRELATIONS WITH MACHINE FAILURE")
print("=" * 50)
print(failure_corr_sorted.head(10))

In [ ]:
# Step 8: Visualize class imbalance
# Machine failure = 1 is only 3.39% of data
# This is critical context — standard accuracy metric is USELESS here
# A model predicting "always 0" gets 96.61% accuracy — but catches NO failures

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Plot 1: Count plot
counts = df['Machine failure'].value_counts()
axes[0].bar(
    ['Normal (0)', 'Failure (1)'],
    counts.values,
    color=['steelblue', 'crimson']
)
axes[0].set_title('Class Distribution (Count)', fontweight='bold')
axes[0].set_ylabel('Number of Records')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 50, str(v), ha='center', fontweight='bold')

# Plot 2: Percentage pie chart
axes[1].pie(
    counts.values,
    labels=[f'Normal\n{counts[0]/len(df)*100:.1f}%',
            f'Failure\n{counts[1]/len(df)*100:.1f}%'],
    colors=['steelblue', 'crimson'],
    autopct='%1.2f%%',
    startangle=90
)
axes[1].set_title('Class Distribution (%)', fontweight='bold')

plt.suptitle('Machine Failure Class Imbalance Analysis',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../plots/class_imbalance.png', dpi=150)
plt.show()

print(f"Normal: {counts[0]} ({counts[0]/len(df)*100:.2f}%)")
print(f"Failure: {counts[1]} ({counts[1]/len(df)*100:.2f}%)")
print(f"Imbalance ratio: {counts[0]/counts[1]:.1f}:1")

In [ ]:
# Step 9: Plot failure type distribution
# Shows which failure mode is most common

failure_flags = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']
failure_counts = [df[f].sum() for f in failure_flags]
failure_labels = [
    'TWF\n(Tool Wear)',
    'HDF\n(Heat Dissipation)',
    'PWF\n(Power)',
    'OSF\n(Overstrain)',
    'RNF\n(Random)'
]

plt.figure(figsize=(10, 6))
bars = plt.bar(failure_labels, failure_counts,
               color=['#e74c3c','#e67e22','#f1c40f','#2ecc71','#3498db'])
plt.title('Distribution of Failure Types', fontsize=14, fontweight='bold')
plt.ylabel('Number of Cases')
plt.xlabel('Failure Type')

for bar, count in zip(bars, failure_counts):
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 1,
             str(count), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../plots/failure_type_distribution.png', dpi=150)
plt.show()

In [ ]:
# Step 10: Tool wear lifecycle chart
# Shows how tool wear accumulates over time and where failures occur
# This captures OPERATIONAL PATTERNS for project documentation

sample = df.head(500).copy()
sample['index'] = range(len(sample))

plt.figure(figsize=(14, 6))
plt.plot(sample['index'],
         sample['Tool wear [min]_roll_mean'],
         color='steelblue', linewidth=1.5,
         label='Tool Wear (Rolling Mean)')

# Mark failure points
failures = sample[sample['Machine failure'] == 1]
plt.scatter(failures['index'],
            failures['Tool wear [min]_roll_mean'],
            color='red', s=100, zorder=5,
            label='Failure Point', marker='X')

plt.title('Tool Wear Lifecycle — Degradation Pattern Over Time',
          fontsize=14, fontweight='bold')
plt.xlabel('Time Sequence (UDI)')
plt.ylabel('Tool Wear Rolling Mean (min)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../plots/tool_wear_lifecycle.png', dpi=150)
plt.show()

print(f"Failures visible in sample: {len(failures)}")

In [ ]:
# Step 11: Compare Torque rolling std between normal and failure cases
# Higher torque instability before failure = key predictive signal

plt.figure(figsize=(8, 5))
sns.boxplot(
    x='Machine failure',
    y='Torque [Nm]_roll_std',
    hue='Machine failure',        # ← ADD THIS (fixes FutureWarning)
    data=df,
    palette={0: 'steelblue', 1: 'crimson'},
    legend=False                  # ← ADD THIS
)
plt.title('Torque Rolling Std: Normal vs Failure',
          fontsize=13, fontweight='bold')
plt.xlabel('Machine Failure (0=Normal, 1=Failure)')
plt.ylabel('Torque Rolling Std')

normal_mean = df[df['Machine failure']==0]['Torque [Nm]_roll_std'].mean()
failure_mean = df[df['Machine failure']==1]['Torque [Nm]_roll_std'].mean()
print(f"Normal avg Torque std  : {normal_mean:.2f}")
print(f"Failure avg Torque std : {failure_mean:.2f}")
print(f"Difference             : {failure_mean - normal_mean:.2f}")

plt.tight_layout()
plt.savefig('../plots/torque_std_by_failure.png', dpi=150)
plt.show()

In [ ]:
# Step 12: EDA Summary — key findings documented
print("=" * 60)
print("TRACK 2 EDA SUMMARY — KEY FINDINGS")
print("=" * 60)
print(f"1. Dataset: 9,996 rows x 40 columns, zero nulls")
print(f"2. Class Imbalance: 96.61% normal, 3.39% failure")
print(f"   → Standard accuracy is misleading — use F1/PR-AUC")
print(f"3. Most common failure: HDF (115) > OSF (98) > PWF (95)")
print(f"4. Torque rolling std: {normal_mean:.2f} normal vs "
      f"{failure_mean:.2f} failure")
print(f"   → Torque instability is a strong failure signal")
print(f"5. Tool wear shows clear degradation pattern before failures")
print("=" * 60)